In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [2]:
df = pd.read_csv('../data/kaggle_b2_fraud_train_v3.csv')
df_test = pd.read_csv('../data/kaggle_b2_fraud_test_v3.csv')

In [30]:
df.columns

Index(['customer_id', 'age', 'tenure_months', 'annual_income_eur',
       'credit_score', 'num_transactions_30d', 'avg_amount_30d_eur',
       'max_amount_30d_eur', 'days_since_last_login', 'support_tickets_90d',
       'chargebacks_12m', 'failed_payments_6m', 'device_trust_z', 'ip_risk_z',
       'is_vpn', 'num_devices_30d', 'is_new_device', 'channel',
       'signup_source', 'plan_type', 'payment_method', 'browser', 'os',
       'occupation', 'device_type', 'merchant_category', 'country', 'region',
       'city', 'target_is_fraud', 'income_log', 'income_estimate_alt_eur',
       'credit_score_norm', 'tx_amount_total_30d_eur', 'max_to_avg_ratio',
       'internal_signal_1', 'internal_signal_2', 'internal_signal_3',
       'internal_signal_4', 'internal_signal_5', 'internal_signal_6',
       'internal_signal_7', 'internal_signal_8', 'terms_accepted_flag',
       'manual_review_result', 'has_second_email',
       'is_missing_last_ticket_subject', 'is_missing_max_amount_30d_eur',
       

## A. Qualité des données / nettoyage logique

### 1.1 Valeurs impossibles détectées

In [3]:
# suppression des âges négatifs
df = df[df['age'] >= 0]
df_test = df_test[df_test['age'] >= 0]

# suppression des dates de création de compte négatives
df = df[df["tenure_months"] >= 0]
df_test = df_test[df_test["tenure_months"] >= 0]

# suppression des revenus négatifs
df = df[df["annual_income_eur"] >= 0]
df_test = df_test[df_test["annual_income_eur"] >= 0]

# suppression des montants négatifs
df = df[df["avg_amount_30d_eur"] >= 0]
df_test = df_test[df_test["avg_amount_30d_eur"] >= 0]

### 1.2 Corrections de typologie nécessaires

In [4]:
# Conversions de types
for dataset in [df, df_test]:
    dataset['is_new_device'] = dataset['is_new_device'].astype('int64')
    dataset['postal_code'] = dataset['postal_code'].astype('string')
    dataset['days_since_last_login'] = dataset['days_since_last_login'].astype('int64')
    dataset['signup_source'] = dataset['signup_source'].astype('object')

### 1.3 suppression doublons inutiles

In [5]:
# suppression des lignes avec des customer_id en double
df = df.drop_duplicates(subset=['customer_id'], keep='first')
df_test = df_test.drop_duplicates(subset=['customer_id'], keep='first')

### 1.4 doublons stricts

In [6]:
df = df.drop_duplicates()
df_test = df_test.drop_duplicates()

## B. Data leakage connu

In [7]:
colonnes_a_supprimer = [
    'chargeback_resolution_time_days',
    'post_event_status_code'
]

df = df.drop(columns=colonnes_a_supprimer)
df_test = df_test.drop(columns=colonnes_a_supprimer)

## C. Identifiants et colonnes inutiles

In [8]:
# Suppression des colonnes avec trop de NaN (>90%)
cols_to_drop = ["legacy_partner_score", "partner_risk_indicator"]

df = df.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

In [9]:
# suppression de 680 lignes avec des customer_id en double ( très peu de changement donc variation)
df = df.drop_duplicates(subset=['customer_id'], keep='first')
df_test = df_test.drop_duplicates(subset=['customer_id'], keep='first')

In [10]:
# trop granulaire, inutile ou non exploitable
cols_to_drop = [
    "referrer_code",   # trop granulaire
    "postal_code",     # trop granulaire : risque overfitting
    "account_id",      # inutile
    "signup_date"      # pas d'informations facilement exploitable
]

df = df.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

In [11]:
# Créer la colonne has_second_email (1 si secondary_email existe, 0 sinon)
df['has_second_email'] = df['secondary_email'].notna().astype(int)
df_test['has_second_email'] = df_test['secondary_email'].notna().astype(int)

# Supprimer la colonne secondary_email
df = df.drop(columns=['secondary_email'])
df_test = df_test.drop(columns=['secondary_email'])

In [12]:
# Créer les flags pour TOUTES les variables avec missingness informatif
for dataset in [df, df_test]:
    dataset['is_missing_last_ticket_subject'] = dataset['last_ticket_subject'].isna().astype(int)
    dataset['is_missing_max_amount_30d_eur'] = dataset['max_amount_30d_eur'].isna().astype(int)

In [13]:
## supprimer les colonnes de texte car on sait pas les traiter
text_cols = ["customer_note","last_ticket_subject"]

df = df.drop(columns=text_cols)
df_test = df_test.drop(columns=text_cols)

## D. Création de features basiques

In [14]:
# création de combinaison nouvelles et logiques

for dataset in [df, df_test]:
    # Interaction binaire × numérique
    dataset["is_new_device_x_num_devices"] = dataset["is_new_device"] * dataset["num_devices_30d"]

    # Interaction binaire × binaire
    dataset["is_vpn_x_ip_risk"] = dataset["is_vpn"] * dataset["ip_risk_z"]

## E. imputations ne dépendant pas de statistiques

In [15]:
df['max_amount_30d_eur'] = df['max_amount_30d_eur'].fillna(0)
df_test['max_amount_30d_eur'] = df_test['max_amount_30d_eur'].fillna(0)

In [16]:
df["ip_risk_z"] = df["ip_risk_z"].fillna(0)
df_test["ip_risk_z"] = df_test["ip_risk_z"].fillna(0)

## Début : Suppression temporaire des features catégorielles à conseerver mais non encodés
## Je les rajouterai petit à petit

In [17]:
# Nombre total de lignes
n_rows = len(df)

# Calcul du nombre et du pourcentage de NaN
missing_df = pd.DataFrame({
    'nb_null': df.isnull().sum(),
    'percent_null': df.isnull().sum() / n_rows * 100
})

# Trier par % décroissant
missing_df = missing_df.sort_values(by='percent_null', ascending=False)
missing_df.head(8)

,nb_null,percent_null
region,38932,28.292988
credit_score,6882,5.001344
device_trust_z,5488,3.988285
occupation,4073,2.959965
is_vpn_x_ip_risk,4069,2.957058
merchant_category,2714,1.972341
avg_amount_30d_eur,0,0.000000
max_amount_30d_eur,0,0.000000


In [18]:
fill_values = {
    'occupation': 'Missing',
    'merchant_category': 'Missing',
    'last_ticket_subject': 'No_ticket',
    'customer_note': 'No_note'
}

df.fillna(value=fill_values, inplace=True)
df_test.fillna(value=fill_values, inplace=True)

# -----------------------------------------------
#                            TRAIN TEST SPLIT
# -----------------------------------------------

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders import TargetEncoder

In [20]:
X = df.drop(columns=['target_is_fraud'])
y = df['target_is_fraud']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

# df_test = fichier Kaggle sans target → on le garde tel quel
X_test = df_test.copy()

Multicolinéarité

In [21]:
# Liste des colonnes à supprimer (VIF trop fort)
col_vif_trop_fort = [
    'terms_accepted_flag',
    'credit_score',
    'income_log',
    'avg_amount_30d_eur',
    'credit_score_norm',
    'income_estimate_alt_eur',
    'max_to_avg_ratio',
    'age',
    'tx_amount_total_30d_eur'
]

# Supprimer les colonnes dans X_train, X_val et X_test
X_train = X_train.drop(columns=col_vif_trop_fort)
X_val = X_val.drop(columns=col_vif_trop_fort)
X_test = X_test.drop(columns=col_vif_trop_fort)

## A. Encodege Catégorielle 

In [22]:

#  device_trust_z → 0
X_train['device_trust_z'] = X_train['device_trust_z'].fillna(0)
X_test['device_trust_z'] = X_test['device_trust_z'].fillna(0)
X_val['device_trust_z'] = X_val['device_trust_z'].fillna(0)


#  is_vpn_x_ip_risk → 0
X_train['is_vpn_x_ip_risk'] = X_train['is_vpn_x_ip_risk'].fillna(0)
X_test['is_vpn_x_ip_risk'] = X_test['is_vpn_x_ip_risk'].fillna(0)
X_val['is_vpn_x_ip_risk'] = X_val['is_vpn_x_ip_risk'].fillna(0)

In [23]:
# Numériques (à scaler)
#num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Colonnes catégorielles
#onehot_cols = ["signup_source","os","browser","device_type","channel","plan_type","country"]
#target_cols = ["payment_method","merchant_category","occupation","city"]


In [24]:
#Numériques (à scaler)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Colonnes catégorielles
onehot_cols = ["signup_source","os","browser","device_type","channel","plan_type","country"]
target_cols = ["payment_method","merchant_category","occupation","city"]

In [25]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # médiane pour credit_score, device_trust_z=0 etc
    ("scaler", RobustScaler())                      # robuste aux outliers
])

onehot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

target_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("target", TargetEncoder())
])

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("onehot", onehot_pipeline, onehot_cols),
        ("target", target_pipeline, target_cols),
    ],
    remainder="drop"
)

In [27]:
X_train_enc = preprocessor.fit_transform(X_train, y_train)
X_val_enc = preprocessor.transform(X_val)

X_test_enc = preprocessor.transform(X_test)


In [28]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['tenure_months', 'annual_income_eur',
                                  'num_transactions_30d', 'max_amount_30d_eur',
                                  'days_since_last_login',
                                  'support_tickets_90d', 'chargebacks_12m',
                                  'failed_payments_6m', 'device_trust_z',
                                  'ip_risk_z', 'is_vpn', 'num_devices_30d',...
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['signup_source', 'os', 'browser',
                                  'device_type', 'channel', 'plan_type',
                                  'country']),
                                ('target',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('target', TargetEncoder())]),
                                 ['payment_method', 'merchant_category',
                                  'occupation', 'city'])])

## B. Feature selection basée sur la corrélation / multicolinéarité

from statsmodels.stats.outliers_influence import variance_inflation_factor

def reduce_vif(X, threshold=5.0):
    X = X.copy()
    while True:
        vif_data = pd.DataFrame()
        vif_data["feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        max_vif = vif_data["VIF"].max()
        if max_vif < threshold:
            break
        # Vérifier importance / corrélation avec y avant suppression si possible
        feature_to_drop = vif_data.sort_values("VIF", ascending=False)["feature"].iloc[0]
        print(f"Supprimer {feature_to_drop} avec VIF={max_vif:.2f}")
        X = X.drop(columns=[feature_to_drop])
    return X

X_train_reduced = reduce_vif(X_train)
X_test_reduced = X_test[X_train_reduced.columns]

| Colonne à supprimer | Colonne conservée |
|---|---|
| `income_estimate_alt_eur` | `annual_income_eur` |
| `income_log` | `annual_income_eur` |
| `credit_score_norm` | `credit_score` |
| `tx_amount_total_30d_eur` | `avg_amount_30d_eur` |
| `is_new_device_x_num_devices` | `is_new_device` |
| `is_vpn_x_ip_risk` | `is_vpn` |
| `city` | `region` |
| `max_to_avg_ratio` | `max_amount_30d_eur` |